In [184]:
import itertools
import numpy as np


In [ ]:
nx, ny, nz = 5, 5, 5
# v_levels = [0]  # 0 for startpoint, 1 for endpoint, 2 for intermediate point
# --- Sample parameter configuration ---
drone_params = {
    'rho': 1.225,      # kg/m3
    'Cd': 0.1,
    'Af': 0.04,        # m2
    'm': 0.991,          # kg
    'g_acc': np.array([0, 0, -9.81]),
    'A': 0.08,          # m2
    'Pp_hover': 150.0  # Watts
}
speed_map = {0: 8.0, 1: 12.0}
dt = 1
dt_takeoff = 1
dt_landing = 1

In [ ]:
def calculate_drone_power(v_i, w_i, a_i, params):
    """
    Calculate total power consumption Pi according to the formula sequence.
    v_i, w_i, a_i are numpy array vectors with shape (3,)
    """
    # Get constants from params
    rho = params['rho']       # Air density
    Cd = params['Cd']         # Drag coefficient
    Af = params['Af']         # Frontal area
    m = params['m']           # Drone weight
    g_acc = params['g_acc']   # Gravitational acceleration vector [0, 0, -9.81]
    A = params['A']           # Rotor disk area
    Pp_hover = params['Pp_hover'] # Profile power when hovering
    mg_scalar = m * np.linalg.norm(g_acc)

    # (2) Velocity relative to wind
    v_a = v_i - w_i
    v_a_norm = np.linalg.norm(v_a)

    # (3) Aerodynamic drag
    Di = -0.5 * rho * Cd * Af * v_a_norm * v_a

    # (4) Thrust force vector
    # print("a_i:", np.array(a_i))
    Ti_vec = m * np.array(a_i) - m * g_acc - Di

    # (5) Magnitude of thrust force
    Ti_mag = np.linalg.norm(Ti_vec)

    # (6) Unit direction vector of thrust
    ti_hat = Ti_vec / Ti_mag if Ti_mag != 0 else np.array([0, 0, 1])

    # (7) Velocity component relative to thrust axis (scalar)
    vc_i = np.dot(v_a, ti_hat)

    # (8) Induced velocity (scalar)
    # Formula: v_ind = -1/2*vc + sqrt((1/2*vc)^2 + Ti/(2*rho*A))
    term1 = -0.5 * vc_i
    term2 = np.sqrt((0.5 * vc_i)**2 + Ti_mag / (2 * rho * A))
    v_ind = term1 + term2

    # (9) Useful power
    Pu_i = np.dot(Ti_vec, v_a)

    # (10) Induced power
    Pind_i = Ti_mag * v_ind

    # (11) Profile power
    Pp_i = Pp_hover * (Ti_mag / mg_scalar)**1.5

    # (12) Total power
    Pi = Pu_i + Pind_i + Pp_i

    return {
        "Pi": Pi,
        "Pu_i": Pu_i,
        "Pind_i": Pind_i,
        "Pp_i": Pp_i,
        "Ti_vec": Ti_vec,
        "Ti_mag": Ti_mag,
    }

In [ ]:

def generate_moore_neighbor_dict(nx, ny, nz,speed_map):
    """
    Create dictionary storing Moore Neighbor edges with determined direction.
    Key format: x1_y1_z1_x2_y2_z2
    """
    moore_dict = {}
    
    # 26 directions of Moore neighborhood (all combinations of -1, 0, 1 except (0,0,0))
    directions = list(itertools.product([-1, 0, 1], repeat=3))
    directions.remove((0, 0, 0))
    
    # Iterate through all vertices in x, y, z space
    for x, y, z in itertools.product(range(nx), range(ny), range(nz)):
        for dx, dy, dz in directions:
            nx_val, ny_val, nz_val = x + dx, y + dy, z + dz
            
            # Check boundaries of 3D space
            if 0 <= nx_val < nx and 0 <= ny_val < ny and 0 <= nz_val < nz:
                # Create key in required format: x1_y1_z1_x2_y2_z2
                for _v in speed_map:  # v=0 for startpoint, v=1 for endpoint
                    edge_key = f"{x}_{y}_{z}_{nx_val}_{ny_val}_{nz_val}_{_v}"
                    distance = np.sqrt((nx_val - x)**2 + (ny_val - y)**2 + (nz_val - z)**2)*10
                    speed = speed_map.get(_v, 5.0)

                    edge_vec = np.array([nx_val - x, ny_val - y, nz_val - z], dtype=float)
                    edge_norm = np.linalg.norm(edge_vec)
                    if edge_norm == 0:
                        continue

                    v_i = (speed / edge_norm) * edge_vec
                    power_result = calculate_drone_power(v_i, [1,1,2], [0,0,0], drone_params)

                    moore_dict[edge_key] = {
                        "startpoint": [x, y, z],
                        "endpoint": [nx_val, ny_val, nz_val],
                        "euclidean_distance": distance,
                        "v_level": _v,
                        "v": speed,
                        "v_vector": v_i,
                        "Pi": power_result["Pi"],
                        "Tmaneuver": power_result["Ti_mag"],
                        "Energy":power_result["Pi"]*(distance/speed)
                        
                    }
                    continue
                
    return moore_dict

In [ ]:
w_i = np.array([1, 1, 2])
space_graph = generate_moore_neighbor_dict(nx, ny, nz,speed_map)

print(f"\nDirected Edge Dictionary: {len(space_graph)}")


Directed Edge Dictionary: 4144


In [ ]:

def calculate_energy_transition(edge_in_key, edge_out_key, wind, params,dt):
    """
    Calculate energy at the intersection point between 2 edges based on Vlevel
    """
    # 1. Decode edge information
    # edge_in_key=space_graph[edge_in_key]
    # edge_out_key=space_graph[edge_out_key]
    p_prev, p_curr, v_in_vec,v_mag_in = edge_in_key["startpoint"], edge_in_key["endpoint"], edge_in_key["v_vector"],edge_in_key["v"]
    _, p_next, v_out_vec,v_mag_out = edge_out_key["startpoint"], edge_out_key["endpoint"], edge_out_key["v_vector"],edge_out_key["v"]

  
    # 3. Calculate v_i and a_i at point p_curr
    # Assume dt is the transition time (average distance / average velocity)
    # dist_avg = (np.linalg.norm(p_curr - p_prev) + np.linalg.norm(p_next - p_curr)) / 2
    # dt = dist_avg / ((v_mag_in + v_mag_out) / 2)

    v_i = (v_in_vec + v_out_vec) / 2
    a_i = (v_out_vec - v_in_vec) / dt

    # --- Physical sequence ---
    rho, Cd, Af, m, A = params['rho'], params['Cd'], params['Af'], params['m'], params['A']
    g_vec = params['g_acc']
    Pp_hover = params['Pp_hover']
    mg_scalar = m * np.linalg.norm(g_vec)

    # (2) Relative velocity
    v_a = v_i - wind
    va_norm = np.linalg.norm(v_a)

    # (3) Drag force Di
    Di = -0.5 * rho * Cd * Af * va_norm * v_a

    # (4) Thrust force Ti (Vector)
    Ti_vec = m * a_i - m * g_vec - Di
    Ti_mag = np.linalg.norm(Ti_vec)

    # (6) Thrust direction
    ti_hat = Ti_vec / Ti_mag if Ti_mag > 1e-6 else np.array([0, 0, 1])

    # (7) Velocity along thrust axis
    vc_i = np.dot(v_a, ti_hat)

    # (8) Induced velocity
    v_ind = -0.5 * vc_i + np.sqrt(max(0, (0.5 * vc_i)**2 + Ti_mag / (2 * rho * A)))

    # (9-12) Power components
    Pu_i = np.dot(Ti_vec, v_a)
    Pind_i = Ti_mag * v_ind
    Pp_i = Pp_hover * (Ti_mag / mg_scalar)**1.5
    Pi = Pu_i + Pind_i + Pp_i

    return {
        # "edge_in": edge_in_key,
        # "edge_out": edge_out_key,
        "total_power_Pi": Pi,
        "Tmaneuver": Ti_mag,
        # "acceleration": a_i
    }


# Drone flies from (1,1,1) -> (2,2,2) with Level 1, then continues to (3,2,2) with Level 0
edge1 = "1_1_1_2_2_2_1"
edge2 = "2_2_2_3_2_2_0"
wind_vec = np.array([1.0, 0, 0]) # Wind blowing along X axis

edge1_data = space_graph[edge1]
edge2_data = space_graph[edge2]
result = calculate_energy_transition(edge1_data, edge2_data, wind_vec, drone_params, dt) # Transition time
print(f"\nEnergy transition from {edge1} to {edge2}: ")
print(f"{result['total_power_Pi']:.2f} J, Thrust force={result['Tmaneuver']:.3f} N")


Energy transition from 1_1_1_2_2_2_1 to 2_2_2_3_2_2_0: 145.11 J, Thrust force=7.495 N


In [190]:
space_graph

{'0_0_0_0_0_1_0': {'startpoint': [0, 0, 0],
  'endpoint': [0, 0, 1],
  'euclidean_distance': 10.0,
  'v_level': 0,
  'v': 8.0,
  'v_vector': array([0., 0., 8.]),
  'Pi': 256.9709449125969,
  'Tmaneuver': 9.812350131576535,
  'Energy': 321.21368114074613},
 '0_0_0_0_0_1_1': {'startpoint': [0, 0, 0],
  'endpoint': [0, 0, 1],
  'euclidean_distance': 10.0,
  'v_level': 1,
  'v': 12.0,
  'v_vector': array([ 0.,  0., 12.]),
  'Pi': 292.48032231434615,
  'Tmaneuver': 9.96920928577898,
  'Energy': 243.7336019286218},
 '0_0_0_0_1_0_0': {'startpoint': [0, 0, 0],
  'endpoint': [0, 1, 0],
  'euclidean_distance': 10.0,
  'v_level': 0,
  'v': 8.0,
  'v_vector': array([0., 8., 0.]),
  'Pi': 208.67108688235038,
  'Tmaneuver': 9.686539097305962,
  'Energy': 260.838858602938},
 '0_0_0_0_1_0_1': {'startpoint': [0, 0, 0],
  'endpoint': [0, 1, 0],
  'euclidean_distance': 10.0,
  'v_level': 1,
  'v': 12.0,
  'v_vector': array([ 0., 12.,  0.]),
  'Pi': 209.2557194840527,
  'Tmaneuver': 9.671479047069942,
  '

In [ ]:
# Filter all edges starting from point (0, 0, 0)
edges_from_origin = {key: value for key, value in space_graph.items() if value['startpoint'] == [0, 0, 0]}

print(f"Number of edges starting from (0,0,0): {len(edges_from_origin)}")
print("\nEdges from origin:")
for key, data in edges_from_origin.items():
    astart = data['v_vector']/dt_takeoff
    Etakeoff = calculate_drone_power(data['v_vector']/2, [1,1,2], astart, drone_params)["Pi"]*dt_takeoff
    space_graph[key]["Etakeoff"] = Etakeoff  # Add takeoff energy to the edge
    print(f"{key}: {data['endpoint']}  Energy={data['Energy']:.2f}, Thrust force={data['Tmaneuver']:.3f} N, Etakeoff={Etakeoff:.2f} J")

Number of edges starting from (0,0,0): 14

Edges from origin:
0_0_0_0_0_1_0: [0, 0, 1]  Energy=321.21, Thrust force=9.812 N, Etakeoff=553.56 J
0_0_0_0_0_1_1: [0, 0, 1]  Energy=243.73, Thrust force=9.969 N, Etakeoff=773.71 J
0_0_0_0_1_0_0: [0, 1, 0]  Energy=260.84, Thrust force=9.687 N, Etakeoff=322.61 J
0_0_0_0_1_0_1: [0, 1, 0]  Energy=174.38, Thrust force=9.671 N, Etakeoff=457.06 J
0_0_0_0_1_1_0: [0, 1, 1]  Energy=425.42, Thrust force=9.776 N, Etakeoff=487.47 J
0_0_0_0_1_1_1: [0, 1, 1]  Energy=310.97, Thrust force=9.882 N, Etakeoff=682.48 J
0_0_0_1_0_0_0: [1, 0, 0]  Energy=260.84, Thrust force=9.687 N, Etakeoff=322.61 J
0_0_0_1_0_0_1: [1, 0, 0]  Energy=174.38, Thrust force=9.671 N, Etakeoff=457.06 J
0_0_0_1_0_1_0: [1, 0, 1]  Energy=425.42, Thrust force=9.776 N, Etakeoff=487.47 J
0_0_0_1_0_1_1: [1, 0, 1]  Energy=310.97, Thrust force=9.882 N, Etakeoff=682.48 J
0_0_0_1_1_0_0: [1, 1, 0]  Energy=368.87, Thrust force=9.689 N, Etakeoff=320.82 J
0_0_0_1_1_0_1: [1, 1, 0]  Energy=246.46, Thrust

In [ ]:
# Filter all edges ending at point (4, 4, 5)
edges_to_destination = {key: value for key, value in space_graph.items() if value['endpoint'] == [4,4,4]}

print(f"Number of edges ending at (4,4,5): {len(edges_to_destination)}")
print("\nEdges to destination:")
for key, data in edges_to_destination.items():
    astart = data['v_vector']/dt_landing
    Elanding = calculate_drone_power(data['v_vector']/2, [1,1,2], astart, drone_params)["Pi"]*dt_landing
    space_graph[key]["Elanding"] = Elanding  # Add landing energy to the edge
    print(f"{key}: {data['endpoint']}  Energy={data['Energy']:.2f}, Thrust force={data['Tmaneuver']:.3f} N, Elanding={Elanding:.2f} J")

Number of edges ending at (4,4,5): 14

Edges to destination:
3_3_3_4_4_4_0: [4, 4, 4]  Energy=506.39, Thrust force=9.759 N, Elanding=456.84 J
3_3_3_4_4_4_1: [4, 4, 4]  Energy=363.68, Thrust force=9.841 N, Elanding=639.99 J
3_3_4_4_4_4_0: [4, 4, 4]  Energy=368.87, Thrust force=9.689 N, Elanding=320.82 J
3_3_4_4_4_4_1: [4, 4, 4]  Energy=246.46, Thrust force=9.673 N, Elanding=453.91 J
3_4_3_4_4_4_0: [4, 4, 4]  Energy=425.42, Thrust force=9.776 N, Elanding=487.47 J
3_4_3_4_4_4_1: [4, 4, 4]  Energy=310.97, Thrust force=9.882 N, Elanding=682.48 J
3_4_4_4_4_4_0: [4, 4, 4]  Energy=260.84, Thrust force=9.687 N, Elanding=322.61 J
3_4_4_4_4_4_1: [4, 4, 4]  Energy=174.38, Thrust force=9.671 N, Elanding=457.06 J
4_3_3_4_4_4_0: [4, 4, 4]  Energy=425.42, Thrust force=9.776 N, Elanding=487.47 J
4_3_3_4_4_4_1: [4, 4, 4]  Energy=310.97, Thrust force=9.882 N, Elanding=682.48 J
4_3_4_4_4_4_0: [4, 4, 4]  Energy=260.84, Thrust force=9.687 N, Elanding=322.61 J
4_3_4_4_4_4_1: [4, 4, 4]  Energy=174.38, Thrust 